# Taller de Clasificación con Árboles de Decisión
**Curso de Inteligencia Artificial**  
*De las preguntas de un árbol a la evaluación de sus predicciones*

---
- **Modalidad:** Individual
- **Arquitectura de Software:** Modular orientada a objetos (SOLID)
- **Librerías principales:** `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `seaborn`
---

## 0. Importación de Módulos y Configuración del Entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn imports
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.dummy import DummyClassifier

# Importar el paquete modular local desarrollado para el taller
from src.synthetic.data import get_diamond_dataset, get_new_cases, get_problem_formulation
from src.synthetic.gini import compute_gini, compare_root_candidates, evaluate_all_possible_splits
from src.synthetic.manual_tree import ManualDiamondTree, get_discussion_section_3
from src.synthetic.metrics import BinaryEvaluationMetrics, analyze_error_costs, analyze_imbalanced_dataset
from src.synthetic.sklearn_verifier import SklearnDiamondVerifier
from src.iris.dataset import IrisDataLoader
from src.iris.experiment import IrisModelSelector
from src.iris.evaluation import IrisTestEvaluator
from src.iris.visualization import IrisVisualizer

print("Entorno modular cargado exitosamente.")

## 1. Formular el problema (0,6 puntos)

### a) Definiciones y justificación de aprendizaje supervisado
- **Unidad de observación:** Un diamante individual presentado para clasificación de corte.
- **Características ($X$):** Vector bidimensional continuo formado por:
  - $x_1$: Quilates (peso del diamante).
  - $x_2$: Profundidad (%) (porcentaje de profundidad respecto al diámetro).
- **Etiqueta ($y$):** Clasificación del corte de la gema, cualitativa y discreta: $y \in \{A, E\}$ donde $A$ = Aceptable y $E$ = Excelente.
- **Clases posibles:** Aceptable ($A$) y Excelente ($E$).
- **Clase positiva convenida:** $A$.
- **¿Por qué es aprendizaje supervisado de clasificación?:**
  Es *supervisado* porque el modelo se entrena a partir de ejemplos históricos donde cada diamante cuenta tanto con sus atributos predictivos $X$ como con su etiqueta objetivo $y$ previamente conocida. Es de *clasificación* porque la variable dependiente $y$ es categórica/discreta (pertenencia a clases cualitativas), a diferencia de la regresión donde se predice una cantidad numérica continua.

### b) Estructura del Árbol de Decisión
- **Raíz (Root):** Nodo inicial (profundidad 0) que agrupa al 100% de los datos de entrenamiento (12 diamantes). Evalúa la primera condición de partición óptima.
- **Nodo Interno (Internal Node):** Cualquier nodo intermedio que evalúa una regla sobre una característica para subdividir un subconjunto impuro.
- **Rama (Branch):** Camino condicional direccional que conecta un nodo con su hijo (izquierda: $x \le t$, derecha: $x > t$).
- **Hoja (Leaf):** Nodo terminal que no tiene descendientes; emite la clasificación final ($A$ o $E$) por mayoría de votos o convención de desempate ($A$).

### c) Análisis de Fuga de Datos (Data Leakage) y Disponibilidad de Variables
- **Campo "resultado final de la inspección":** NO debe utilizarse. Al registrarse con posterioridad o en simultáneo a la emisión de la clase, constituye una **fuga de datos (data leakage)**. En el momento real de la predicción previa este campo no existirá; incluirlo provocaría un modelo con desempeño artificialmente perfecto en entrenamiento que fallará totalmente en producción.
- **Condición para incorporar el Precio:** El precio debe estar disponible y consolidado **antes** de emitir la clasificación y no debe ser una consecuencia de la clase asignada (evitar dependencia circular).

In [ ]:
# Visualización del conjunto de datos sintético
df_diamantes = get_diamond_dataset()
print("Conjunto de datos sintético (12 diamantes):")
display(df_diamantes)

## 2. Seleccionar la raíz con el índice Gini (1,2 puntos)

### Fórmulas Matemáticas:
Para un nodo $S$ con $K$ clases:
$$G(S) = 1 - \sum_{k=1}^K p_k^2$$

Impureza ponderada de una división sobre los hijos $S_L$ y $S_R$:
$$G_{div} = \frac{|S_L|}{|S|} G(S_L) + \frac{|S_R|}{|S|} G(S_R)$$

Reducción de impureza (ganancia Gini):
$$\Delta G = G(S) - G_{div}$$

### a) Impureza de la Raíz
El conjunto total $S$ tiene $|S| = 12$ diamantes ($n_A = 6, n_E = 6$).
$$p_A = \frac{6}{12} = 0.5000, \quad p_E = \frac{6}{12} = 0.5000$$
$$G(S) = 1 - (0.5000^2 + 0.5000^2) = 1 - (0.2500 + 0.2500) = 0.5000$$

### b) Comparación de Preguntas Candidatas para la Raíz
Se comparan exclusivamente:
1. `Quilates <= 0.75`
2. `Profundidad <= 63.5`

In [ ]:
# Cálculo exacto de las preguntas candidatas a la raíz
summary_df, results = compare_root_candidates(df_diamantes)
print("=== TABLA COMPARATIVA DE CANDIDATOS A LA RAÍZ ===")
display(summary_df)

### c) Elección y Justificación de la Raíz
- **Pregunta Ganadora:** Se elige **`Profundidad <= 63.5`** debido a que produce una reducción de impureza de $\Delta G = 0.2500$, muy superior a $\Delta G = 0.1286$ de `Quilates <= 0.75`. Además, aísla un nodo derecho totalmente puro ($G(S_R) = 0.0000$, con 4 muestras de clase A).
- **¿Por qué no promediar sin considerar tamaños?:** Porque las particiones poseen distinto número de elementos ($|S_L| \neq |S_R|$). Un promedio simple distorsionaría el valor esperado de impureza, dándole el mismo peso a una hoja con 1 observación que a una con 11 observaciones.
- **No optimalidad global:** Comparar solo 2 preguntas candidatas no garantiza optimalidad global porque existen múltiples puntos medios entre los valores continuos de ambas variables que no fueron evaluados.

## 3. Construir e interpretar el árbol (1,0 punto)

### a) Partición del Hijo Impuro
El hijo izquierdo de la raíz ($S_L$: `Profundidad <= 63.5`) contiene 8 diamantes (IDs: 1, 2, 4, 6, 7, 9, 11, 12) con 2 A y 6 E ($G = 0.3750$).
Ordenando los valores de Quilates en $S_L$: `[0.3, 0.4, 0.6, 0.8, 0.9, 1.1, 1.3, 1.4]`.
El punto medio entre 0.4 y 0.6 es **`Quilates <= 0.50`**:
- $S_{LL}$ (Quilates $\le 0.50$): IDs [1, 2] $\rightarrow$ 2 A, 0 E. **$G(S_{LL}) = 0.0000$ (Puro, Clase A)**.
- $S_{LR}$ (Quilates $> 0.50$): IDs [4, 6, 7, 9, 11, 12] $\rightarrow$ 0 A, 6 E. **$G(S_{LR}) = 0.0000$ (Puro, Clase E)**.
$$G_{div} = \frac{2}{8}(0.0000) + \frac{6}{8}(0.0000) = 0.0000 \implies \Delta G = 0.3750 - 0.0000 = 0.3750$$
¡Ambas hojas resultan 100% puras!

In [ ]:
# Instanciar el árbol manual y visualizar su arquitectura
manual_tree = ManualDiamondTree()
print("Reglas del árbol manual:")
for r in manual_tree.rules:
    print(f"• {r['id']}: {r['condition']} -> Clase {r['clase']} ({r['soporte']})")

# Generar y mostrar el diagrama del árbol manual
fig_path = manual_tree.plot_tree_diagram()
img = plt.imread(fig_path)
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.title("Estructura Arquitectónica del Árbol Manual", fontsize=14, weight='bold')
plt.show()

### b) Clasificación y Trazabilidad de Casos Nuevos (N1 a N4)

In [ ]:
df_nuevos = get_new_cases()
df_pred_manual = manual_tree.predict_dataframe(df_nuevos)
print("=== CLASIFICACIÓN DE CASOS NUEVOS (N1 - N4) ===")
display(df_pred_manual[["Caso", "Quilates", "Profundidad", "Prediccion_Manual", "Hoja_Asignada", "Recorrido_Seguido"]])

### c) Discusión sobre Validación y Capacidad de Generalización
- **¿Puede afirmarse que acertó en N1-N4?:** No. Los casos N1 a N4 no disponen de etiqueta real conocida (sin ground truth), por lo que es imposible verificar matemáticamente si la predicción es correcta o errada.
- **¿Qué demuestra un 100% de exactitud sobre las 12 filas de entrenamiento?:** Demuestra únicamente que el modelo tiene capacidad representativa suficiente para memorizar o separar perfectamente los datos observados de la muestra (ajuste perfecto muestral).
- **¿Qué NO demuestra?:** NO demuestra capacidad de generalización hacia nuevos diamantes fuera de la muestra. En conjuntos tan pequeños, lograr 100% de precisión en entrenamiento conlleva un alto riesgo de sobreajuste (overfitting).

## 4. Evaluar las predicciones (0,8 puntos)

### a) Evaluación sobre conjunto independiente de 20 casos
Matriz de confusión observada:
- Real A, Predicho A: **$VP = 6$**
- Real A, Predicho E: **$FN = 2$**
- Real E, Predicho A: **$FP = 3$**
- Real E, Predicho E: **$VN = 9$**
- Total: $N = 20$

In [ ]:
metrics = BinaryEvaluationMetrics(vp=6, vn=9, fp=3, fn=2)
m_results = metrics.calculate_all()

rows_m = []
for m in ["Exactitud", "Precision", "Recall", "F1"]:
    d = m_results[m]
    rows_m.append({
        "Métrica": m,
        "Fórmula": d["formula"],
        "Sustitución": d["sustitucion"],
        "Resultado": f"{d['valor']:.4f} ({d['porcentaje']})",
        "Interpretación": d["interpretacion"]
    })
display(pd.DataFrame(rows_m))

### b) Análisis Asimétrico de Costos
- **Excelente clasificado como Aceptable (FP respecto a A):** En un escenario de venta comercial por mayor, vender diamantes Excelentes al precio de Aceptables genera una pérdida financiera directa de ingresos por gema (lucro cesante).
- **Aceptable clasificado como Excelente (FN respecto a A):** En un laboratorio certificador o joyería de alta gama, emitir un certificado de grado 'Excelente' para un diamante imperfecto provocará litigios por estafa, multas regulatorias y destrucción total del prestigio de la marca.

In [ ]:
# Análisis del conjunto desbalanceado (95 A y 5 E)
imb = analyze_imbalanced_dataset()
print("=== PARADOJA DE LA EXACTITUD EN CONJUNTO DESBALANCEADO ===")
print(f"Modelo: {imb['modelo']}")
print(f"Exactitud Global: {imb['exactitud']['valor']:.4f} ({imb['exactitud']['porcentaje']})")
print(f"Recall de Clase E: {imb['recall_E']['valor']:.4f} ({imb['recall_E']['porcentaje']})")
print(f"\nConclusión:\n{imb['justificacion']}")

## 5. Implementación y generalización en Python (1,4 puntos)

### Parte A: Verificar el cálculo manual con Scikit-Learn (0,3 puntos)

In [ ]:
verifier = SklearnDiamondVerifier()
verifier.fit()
comp_df = verifier.compare_predictions()
print("Comparación de predicciones Manual vs Scikit-Learn:")
display(comp_df[["Caso", "Quilates", "Profundidad", "Prediccion_Manual", "Prediccion_Sklearn", "Coincide"]])

# Visualizar el árbol de scikit-learn
sk_fig = verifier.plot_tree_sklearn()
img_sk = plt.imread(sk_fig)
plt.figure(figsize=(10, 6))
plt.imshow(img_sk)
plt.axis('off')
plt.title("Árbol de Decisión Scikit-Learn (max_depth=2, random_state=42)", fontsize=14, weight='bold')
plt.show()

### Parte B: Experimento con Iris (1,1 puntos)

#### a) Carga de datos y partición estratificada 60%/20%/20% con semilla fija

In [ ]:
iris_loader = IrisDataLoader()
X_train, X_val, X_test, y_train, y_val, y_test = iris_loader.get_partitions()

dist_df = iris_loader.get_partition_distribution()
print("Distribución estratificada de clases por partición:")
display(dist_df)

#### b) Entrenamiento de candidatos y selección sobre validación

In [ ]:
selector = IrisModelSelector(iris_loader)
res_table = selector.run_training_and_validation()
best_name, best_model, best_info = selector.select_best_model()

print("=== TABLA COMPARATIVA DE CANDIDATOS (ENTRENAMIENTO VS VALIDACIÓN) ===")
display(res_table)
print(f"\n--> MODELO GANADOR SELECCIONADO: {best_name}")

#### c) Evaluación final e independiente en Prueba (Test Set) y Benchmark contra Dummy

In [ ]:
evaluator = IrisTestEvaluator(best_model, iris_loader)
test_results = evaluator.evaluate_test_set()
benchmark_df = evaluator.benchmark_against_dummy()

print("=== BENCHMARK: MODELO SELECCIONADO VS DUMMY CLASSIFIER EN PRUEBA ===")
display(benchmark_df)

print("\n=== REPORTE DE CLASIFICACIÓN DETALLADO EN PRUEBA ===")
print(test_results["classification_report"])

#### d) Visualización del Árbol Seleccionado, Matriz de Confusión y Explicación de Caminos

In [ ]:
visualizer = IrisVisualizer(best_model, iris_loader)

# Graficar árbol seleccionado
fig_tree_iris = visualizer.plot_selected_tree()
img_iris = plt.imread(fig_tree_iris)
plt.figure(figsize=(14, 8))
plt.imshow(img_iris)
plt.axis('off')
plt.title("Árbol Seleccionado (max_depth=None, Efectiva=4)", fontsize=14, weight='bold')
plt.show()

# Graficar Matriz de Confusión
fig_cm = visualizer.plot_confusion_matrix(test_results["confusion_matrix"])
img_cm = plt.imread(fig_cm)
plt.figure(figsize=(7, 5))
plt.imshow(img_cm)
plt.axis('off')
plt.show()

# Explicar dos caminos
for p in visualizer.explain_two_paths():
    print(f"\n--- {p['nombre']} ---")
    print(f"Recorrido: {p['recorrido']}")
    print(f"Soporte: {p['soporte']}")
    print(f"Interpretación: {p['interpretacion']}")

## 6. Preguntas para Socializar

1. **¿Por qué una hoja pura no garantiza aciertos futuros?**  
   Una hoja pura garantiza únicamente que el conjunto de entrenamiento no tiene error en esa región. Los datos futuros pueden contener ruido estocástico o solapamiento natural de clases. Cuando una hoja pura se forma con pocas muestras (ej. 1 o 2 observaciones), suele responder a memorización de ruido y sobreajuste.

2. **¿Qué cambia al limitar la profundidad?**  
   Limitar la profundidad (`max_depth`) actúa como poda previa (pre-pruning), controlando la varianza del modelo a cambio de un ligero sesgo, lo cual previene el sobreajuste y fomenta árboles parsimoniosos y generalizables.

3. **¿Por qué F1 macro resulta útil al comparar las tres especies?**  
   Porque evalúa la media no ponderada del F1 de cada una de las 3 especies, dándole el mismo valor a cada clase y evitando que el buen rendimiento en una especie mayoritaria enmascare errores en especies minoritarias.

4. **¿Qué información adicional necesitaría para usar el modelo fuera del taller?**  
   Se requiere: matriz de costos comerciales, variables industriales completas (color, claridad, simetría para diamantes), calibración de probabilidades para emitir umbrales de incertidumbre y protocolos de monitoreo de deriva de datos (data drift).